In [ ]:
import os
import numpy as np
from PIL import Image
import random
import hashlib
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
# Reproducibility
SEED = 12345
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Dataset settings
classes = ['cross', 'square', 'triangle', 'x']
label_map = {cls: i for i, cls in enumerate(classes)}

samples_per_class = 100
output_dir = 'generated_icons_v2'
os.makedirs(output_dir, exist_ok=True)

IMG_SIZE = 4
NUM_PIXELS = IMG_SIZE * IMG_SIZE

In [ ]:
# Grayscale intensity settings
BACKGROUND_MIN = 0
BACKGROUND_MAX = 50

SHAPE_MIN = 180
SHAPE_MAX = 255

TEST_SIZE = 0.3
SPLIT_SEED = 246

In [ ]:
# Utility functions
def render_from_mask(mask):
    """
    Convert a binary shape mask into a grayscale image.

    Background pixels receive small random noise.
    Shape pixels receive high intensity values.
    """
    img = np.random.randint(
        BACKGROUND_MIN,
        BACKGROUND_MAX + 1,
        size=(IMG_SIZE, IMG_SIZE),
        dtype=np.uint8
    )

    shape_values = np.random.randint(
        SHAPE_MIN,
        SHAPE_MAX + 1,
        size=(IMG_SIZE, IMG_SIZE),
        dtype=np.uint8
    )

    img[mask == 1] = shape_values[mask == 1]
    return img

In [ ]:
def hash_image(img):
    """Create a hash to avoid exact duplicate images."""
    return hashlib.sha256(img.tobytes()).hexdigest()

In [ ]:
def grayscale_to_angle(df):
    """
    Convert grayscale pixel values from [0, 255]
    to quantum rotation angles in [0, pi/2].
    """
    angle_df = df.copy()
    pixel_cols = [f'pix{i}' for i in range(NUM_PIXELS)]

    angle_df[pixel_cols] = angle_df[pixel_cols].astype(float) / 255.0 * (np.pi / 2)
    return angle_df

In [ ]:
# Shape generators
def generate_cross():
    """
    Generate a thin cross in a 4x4 grid.
    Row and column positions vary independently.
    """
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    row_center = random.choice([1, 2])
    col_center = random.choice([1, 2])

    mask[row_center, :] = 1
    mask[:, col_center] = 1

    return render_from_mask(mask)

In [ ]:
def generate_square():
    """
    Generate an outline square in a 4x4 grid.
    """
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    size = random.choice([3, 4])
    max_start = IMG_SIZE - size

    start_x = random.randint(0, max_start)
    start_y = random.randint(0, max_start)

    # top and bottom edges
    mask[start_y, start_x:start_x + size] = 1
    mask[start_y + size - 1, start_x:start_x + size] = 1

    # left and right edges
    mask[start_y:start_y + size, start_x] = 1
    mask[start_y:start_y + size, start_x + size - 1] = 1

    return render_from_mask(mask)

In [ ]:
def generate_triangle():
    """
    Generate an outline triangle in a 4x4 grid.
    Supports four orientations: up, down, left, right.
    """
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    orientation = random.choice(['up', 'down', 'left', 'right'])

    if orientation == 'up':
        coords = [
            (0, 1), (0, 2),
            (1, 1), (1, 2),
            (2, 0), (2, 1), (2, 2), (2, 3)
        ]

    elif orientation == 'down':
        coords = [
            (1, 0), (1, 1), (1, 2), (1, 3),
            (2, 1), (2, 2),
            (3, 1), (3, 2)
        ]

    elif orientation == 'left':
        coords = [
            (1, 0), (2, 0),
            (1, 1), (2, 1),
            (0, 2), (1, 2), (2, 2), (3, 2)
        ]

    else:  # right
        coords = [
            (0, 1), (1, 1), (2, 1), (3, 1),
            (1, 2), (2, 2),
            (1, 3), (2, 3)
        ]

    for y, x in coords:
        mask[y, x] = 1

    return render_from_mask(mask)

In [ ]:
def generate_x():
    """
    Generate an X shape in a 4x4 grid.
    Variants include full 4x4 X and smaller 3x3 X with position variation.
    """
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

    variant = random.choice(['full', 'small'])

    if variant == 'full':
        size = 4
        start_y = 0
        start_x = 0
    else:
        size = 3
        start_y = random.choice([0, 1])
        start_x = random.choice([0, 1])

    for i in range(size):
        mask[start_y + i, start_x + i] = 1
        mask[start_y + i, start_x + (size - 1 - i)] = 1

    return render_from_mask(mask)

In [ ]:
generators = {
    'cross': generate_cross,
    'square': generate_square,
    'triangle': generate_triangle,
    'x': generate_x
}

In [ ]:
# Dataset generation
data = []
labels = []
label_ids = []
filenames = []
hashes = set()

for cls in classes:
    count = 0
    tries = 0
    max_tries = 5000

    while count < samples_per_class and tries < max_tries:
        img = generators[cls]()
        img_hash = hash_image(img)
        tries += 1

        if img_hash not in hashes:
            hashes.add(img_hash)

            filename = f"{cls}_{count:03}.png"
            filepath = os.path.join(output_dir, filename)
            Image.fromarray(img).save(filepath)

            data.append(img.flatten())
            labels.append(cls)
            label_ids.append(label_map[cls])
            filenames.append(filename)

            count += 1

    if count < samples_per_class:
        print(f"Warning: only generated {count} unique samples for class '{cls}'.")

In [ ]:
# Save grayscale dataset
pixel_cols = [f'pix{i}' for i in range(NUM_PIXELS)]

df_gray = pd.DataFrame(data, columns=pixel_cols)
df_gray.insert(0, 'filename', filenames)
df_gray.insert(1, 'label', labels)
df_gray.insert(2, 'label_id', label_ids)

gray_path = os.path.join(output_dir, 'greyscale_dataset.csv')
df_gray.to_csv(gray_path, index=False)

In [ ]:
# Save angle dataset for QCNN input
df_angle = grayscale_to_angle(df_gray)

angle_path = os.path.join(output_dir, 'angle_dataset.csv')
df_angle.to_csv(angle_path, index=False)

In [ ]:
# Train/test split
gray_train, gray_test = train_test_split(
    df_gray,
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=df_gray['label']
)

angle_train, angle_test = train_test_split(
    df_angle,
    test_size=TEST_SIZE,
    random_state=SPLIT_SEED,
    stratify=df_angle['label']
)

gray_train = gray_train.reset_index(drop=True)
gray_test = gray_test.reset_index(drop=True)
angle_train = angle_train.reset_index(drop=True)
angle_test = angle_test.reset_index(drop=True)

gray_train.to_csv(os.path.join(output_dir, 'greyscale_train.csv'), index=False)
gray_test.to_csv(os.path.join(output_dir, 'greyscale_test.csv'), index=False)

angle_train.to_csv(os.path.join(output_dir, 'angle_train.csv'), index=False)
angle_test.to_csv(os.path.join(output_dir, 'angle_test.csv'), index=False)

# Zipping the directory
shutil.make_archive(output_dir, 'zip', output_dir)

In [ ]:
# Summary
print("Dataset generation complete.")
print(f"Output directory: {output_dir}")
print(f"Total samples: {len(df_gray)}")

print("\nClass distribution:")
print(df_gray['label'].value_counts())

print("\nSaved files:")
print("- greyscale_dataset.csv")
print("- angle_dataset.csv")
print("- greyscale_train.csv")
print("- greyscale_test.csv")
print("- angle_train.csv")
print("- angle_test.csv")